# Strategy & Governance Indicator: 10-K AI Sentiment

Sentiment of AI-related language in Fortune 500 10-K filings, scored
with FinBERT-tone (Huang, Wang, Yang 2023, CAR). The keyword list
takes Basnet (2025, "Analyzing the market's reaction to AI narratives
in corporate filings") as its canonical 18-term base and extends it
with modern terminology covering generative AI, foundation models,
agentic systems, and modern learning paradigms.

**Pipeline.** Resolve Fortune 500 tickers to EDGAR CIKs, download
the most recent 10-K per firm, extract Items 1, 1A, and 7, filter
sentences against the AI keyword list, score each AI-relevant sentence
with FinBERT-tone, and aggregate to one row per firm. The output is
written to `data_clean/indicators/strategy_governance.parquet`.

Item 7A (Quantitative and Qualitative Disclosures About Market Risk)
is deliberately excluded: dominated by tabular financial-market risk
content, Loughran-McDonald (2016, JAR) flag it as unreliable for tone
analysis, and Babina et al. (2024) report near-zero AI mentions there.

**Prerequisites.**
- The conda environment from `environment.yml` is active (provides
  `spacy` + `en_core_web_sm`, `transformers`, `edgartools`, etc.).
- `EDGAR_IDENTITY` environment variable set to `"Your Name email@host"`.
  The setup cell below falls back to a hardcoded default if it is unset.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import logging
import os
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.strategy_governance import (
    AI_KEYWORDS,
    aggregate_firm_level,
    extract_items,
    filter_ai_sentences,
    resolve_fortune500_filings,
    score_sentences,
)
from src.indicators.strategy_governance.aggregate import MIN_AI_SENTENCES_FOR_TONE
from src.indicators.strategy_governance.filter import (
    aggregate_keyword_counts,
    count_keyword_occurrences,
    is_ai_sentence,
    split_sentences,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

os.environ.setdefault("EDGAR_IDENTITY", "Timo Koba kab.timo3@gmail.com")

# Each slow step (filings, sentences, sentence_totals, keyword_distribution, scored)
# is cached under data_cache/indicators/strategy_governance/. Toggle
# FORCE_REFRESH to True to recompute everything from scratch. Or call
# clear_cache("strategy_governance") / clear_cache("strategy_governance", "<step>")
# to invalidate selectively.
FORCE_REFRESH = False
INDICATOR = "strategy_governance"

# Validation-sample knobs (section 3). Surface here so a reviewer can find
# them without scrolling into the validation cells.
VALIDATION_DIR = PROJECT_ROOT / "data_clean" / "validation" / INDICATOR
ANNOTATION_FILE = VALIDATION_DIR / "sample_to_annotate.csv"
KEY_FILE = VALIDATION_DIR / "_key.parquet"
SAMPLE_FILINGS_N = 30
POS_N = 100
NEG_N = 100
SEED = 42

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Caches cleared. Ready to re-run.


## 1. Resolve filings

Look up the latest 10-K for each Fortune 500 ticker. Foreign filers
(20-F) and resolution failures are dropped and logged.

In [3]:
filings = None if FORCE_REFRESH else load_cached_step(INDICATOR, "filings")
if filings is None:
    filings = resolve_fortune500_filings(fiscal_year=None)
    save_cached_step(filings, INDICATOR, "filings")
    print(f"Resolved {len(filings)} 10-K filings (saved to {cache_path(INDICATOR, 'filings')})")
else:
    print(f"Loaded {len(filings)} 10-K filings from cache ({cache_path(INDICATOR, 'filings')})")
filings.head()

Loaded 453 10-K filings from cache (D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\strategy_governance\filings.parquet)


,cik,ticker,company_name,normalized_company_name,accession_number,fiscal_year,filing_date,form
0,0000104169,WMT,Walmart,walmart,0000104169-26-000055,2026,2026-03-13,10-K
1,0001018724,AMZN,Amazon,amazon,0001018724-26-000004,2025,2026-02-06,10-K
2,0000731766,UNH,UnitedHealth Group,unitedhealth,0000731766-26-000062,2025,2026-03-02,10-K
3,0000320193,AAPL,Apple,apple,0000320193-25-000079,2025,2025-10-31,10-K
4,0000064803,CVS,CVS Health,cvs health,0000064803-26-000010,2025,2026-02-10,10-K


## 2. Parse Item sections + filter AI sentences

Iterate over filings, pull Items 1, 1A, and 7, sentence-tokenize, and
split each filing's sentences into AI-relevant (matched against
`AI_KEYWORDS`) and total cleaned counts. The cleaned-sentence totals
become the denominator of `ai_sentence_share` in section 5
(length-normalized AI disclosure intensity, Loughran-McDonald
2011 convention).

In [4]:
print(f"AI_KEYWORDS: {len(AI_KEYWORDS)} keywords")

sentences_df = None if FORCE_REFRESH else load_cached_step(INDICATOR, "sentences")
sentence_totals_df = None if FORCE_REFRESH else load_cached_step(INDICATOR, "sentence_totals")
keyword_distribution_df = None if FORCE_REFRESH else load_cached_step(INDICATOR, "keyword_distribution")

if sentences_df is None or sentence_totals_df is None or keyword_distribution_df is None:
    all_sentences: list[pd.DataFrame] = []
    totals_rows: list[dict] = []
    errors: list[tuple[str, str]] = []
    kw_counter: Counter = Counter()
    for _, row in filings.iterrows():
        try:
            sections = extract_items(row["accession_number"])
            sub, totals = filter_ai_sentences(
                sections.sections,
                accession_number=row["accession_number"],
                cik=row["cik"],
            )
            if len(sub) > 0:
                all_sentences.append(sub)
            totals_rows.append(
                {
                    "accession_number": row["accession_number"],
                    "cik": row["cik"],
                    "n_sentences_item_1": int(totals.get("item_1", 0)),
                    "n_sentences_item_1a": int(totals.get("item_1a", 0)),
                    "n_sentences_item_7": int(totals.get("item_7", 0)),
                }
            )
            for _, text in sections.sections.items():
                kw_counter.update(count_keyword_occurrences(text))
        except Exception as exc:
            errors.append((row["accession_number"], str(exc)))

    sentences_df = pd.concat(all_sentences, ignore_index=True) if all_sentences else pd.DataFrame()
    save_cached_step(sentences_df, INDICATOR, "sentences")

    sentence_totals_df = pd.DataFrame(totals_rows)
    save_cached_step(sentence_totals_df, INDICATOR, "sentence_totals")

    keyword_distribution_df = aggregate_keyword_counts(kw_counter)
    save_cached_step(keyword_distribution_df, INDICATOR, "keyword_distribution")

    print(f"AI-relevant sentences: {len(sentences_df)} (saved to {cache_path(INDICATOR, 'sentences')})")
    print(f"Sentence totals saved to {cache_path(INDICATOR, 'sentence_totals')}")
    print(f"Keyword distribution saved to {cache_path(INDICATOR, 'keyword_distribution')}")
    print(f"Parse errors: {len(errors)}")
else:
    print(f"Loaded {len(sentences_df)} AI sentences from cache ({cache_path(INDICATOR, 'sentences')})")
    print(f"Loaded sentence totals from cache ({cache_path(INDICATOR, 'sentence_totals')})")
    print(f"Loaded keyword distribution from cache ({cache_path(INDICATOR, 'keyword_distribution')})")

print(f"Filings with at least one AI mention: {sentences_df['accession_number'].nunique() if len(sentences_df) else 0}")
print(f"Keyword concept groups: {len(keyword_distribution_df)}")
print(f"Groups with 0 occurrences: {int((keyword_distribution_df['count'] == 0).sum())}")
print("\nTop 15 concept groups by total occurrences:")
print(keyword_distribution_df.head(15).to_string(index=False))

AI_KEYWORDS: 80 keywords


2026-05-13 15:59:47,980 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 15:59:53,473 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 15:59:53,474 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 15:59:59,997 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 16:00:02,609 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 16:00:02,609 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 16:00:04,962 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 16:00:10,666 INFO edgar.documents.extractors.toc_section_detector TOC detection found 27 sections
2026-05-13 16:00:10,667 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 27 sections found


2026-05-13 16:00:13,856 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 16:00:25,077 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 16:00:25,081 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


...

2026-05-13 17:29:26,567 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:29:29,099 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:29:29,100 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:29:32,466 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:29:38,250 INFO edgar.documents.extractors.toc_section_detector TOC detection found 25 sections
2026-05-13 17:29:38,250 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 25 sections found


2026-05-13 17:29:41,389 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:29:45,883 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:29:45,899 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:29:48,488 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:29:53,317 INFO edgar.documents.extractors.toc_section_detector TOC detection found 21 sections
2026-05-13 17:29:53,317 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 21 sections found


AI-relevant sentences: 6498 (saved to D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\strategy_governance\sentences.parquet)
Sentence totals saved to D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\strategy_governance\sentence_totals.parquet
Keyword distribution saved to D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\strategy_governance\keyword_distribution.parquet
Parse errors: 1
Filings with at least one AI mention: 418
Keyword concept groups: 58
Groups with 0 occurrences: 20

Top 15 concept groups by total occurrences:
                           keyword  count  share_pct
                                ai   6563      69.87
           artificial intelligence   1445      15.38
                  machine learning    471       5.01
                     generative ai    363       3.86
                    

## 3. Validation: keyword dictionary precision / recall / F1 *(optional, for appendix)*

Stratified random sample of 200 sentences (100 dict-positive, 100 dict-negative) drawn from a 30-filing subsample. Sentences are shuffled and the source label is hidden so you can annotate blind.

**Workflow.**
1. Run the **Build sample** cell once. It writes `data_clean/validation/strategy_governance/sample_to_annotate.csv`.
2. Open that CSV in Excel (or any CSV editor). Fill in the `gold` column with `1` if the sentence substantively discusses AI/ML technology, deployment, governance, or strategy, else `0`. Be strict - borderline cases default to `0`. Save (keep the same filename).
3. Run the **Compute metrics** cell. It joins your annotations with the hidden source key and prints precision, recall, F1 plus example errors.

To regenerate the sample, delete the CSV.

In [5]:
# Self-load `filings` from cache if it isn't already in memory (so this cell
# can run after a kernel restart without re-executing section 1).
try:
    filings
except NameError:
    filings = load_cached_step(INDICATOR, "filings")
    if filings is None:
        raise RuntimeError(
            "No cached filings found. Run section 1 (Resolve filings) first."
        )
    print(f"Loaded {len(filings)} filings from cache for validation")

if ANNOTATION_FILE.exists():
    print(f"Sample already exists: {ANNOTATION_FILE}")
    print("Delete the file to regenerate, or annotate it and run the metrics cell.")
else:
    pool_filings = filings.sample(n=min(SAMPLE_FILINGS_N, len(filings)), random_state=SEED)
    pool_pos: list[dict] = []
    pool_neg: list[dict] = []
    for _, row in pool_filings.iterrows():
        try:
            sections = extract_items(row["accession_number"])
        except Exception:
            continue
        for item, text in sections.sections.items():
            for sent in split_sentences(text):
                rec = {"sentence": sent, "item": item, "accession_number": row["accession_number"]}
                (pool_pos if is_ai_sentence(sent) else pool_neg).append(rec)

    rng = random.Random(SEED)
    rng.shuffle(pool_pos)
    rng.shuffle(pool_neg)
    selected = pd.DataFrame(
        [{**r, "source": "pos"} for r in pool_pos[:POS_N]]
        + [{**r, "source": "neg"} for r in pool_neg[:NEG_N]]
    )
    selected = selected.sample(frac=1, random_state=SEED).reset_index(drop=True)
    selected.insert(0, "id", range(len(selected)))

    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    selected[["id", "source"]].to_parquet(KEY_FILE, index=False)
    annotation = selected[["id", "sentence", "item", "accession_number"]].copy()
    annotation["gold"] = ""
    annotation.to_csv(ANNOTATION_FILE, index=False, encoding="utf-8-sig")

    print(f"Pool sizes: {len(pool_pos)} positives / {len(pool_neg)} negatives across {len(pool_filings)} filings")
    print(f"Wrote {len(annotation)} sentences to: {ANNOTATION_FILE}")
    print("Open it in Excel, fill the 'gold' column with 1 or 0, save, then run the next cell.")

2026-05-13 17:29:57,342 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:30:02,017 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:30:02,017 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:30:04,801 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:30:10,708 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:30:10,709 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:30:16,114 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:30:19,166 INFO edgar.documents.extractors.toc_section_detector TOC detection found 25 sections
2026-05-13 17:30:19,167 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 25 sections found


2026-05-13 17:30:23,143 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:30:25,684 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:30:25,684 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:30:30,434 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:30:37,068 INFO edgar.documents.extractors.toc_section_detector TOC detection found 13 sections
2026-05-13 17:30:37,068 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 13 sections found


...

2026-05-13 17:34:02,686 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:34:06,969 INFO edgar.documents.extractors.toc_section_detector TOC detection found 23 sections
2026-05-13 17:34:06,969 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-05-13 17:34:10,268 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:34:13,186 INFO edgar.documents.extractors.toc_section_detector TOC detection found 24 sections
2026-05-13 17:34:13,186 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 24 sections found


2026-05-13 17:34:15,635 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:34:17,219 INFO edgar.documents.extractors.toc_section_detector TOC detection found 11 sections
2026-05-13 17:34:17,219 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 11 sections found


2026-05-13 17:34:20,336 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]
2026-05-13 17:34:25,186 INFO edgar.documents.extractors.toc_section_detector TOC detection found 22 sections
2026-05-13 17:34:25,186 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 22 sections found


Pool sizes: 556 positives / 33144 negatives across 30 filings
Wrote 200 sentences to: d:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_clean\validation\strategy_governance\sample_to_annotate.csv
Open it in Excel, fill the 'gold' column with 1 or 0, save, then run the next cell.


In [6]:
if not ANNOTATION_FILE.exists() or not KEY_FILE.exists():
    raise RuntimeError(
        f"Validation files not found under {VALIDATION_DIR}. "
        "Run the Build sample cell first."
    )

annotated = pd.read_csv(ANNOTATION_FILE, encoding="utf-8-sig")
key = pd.read_parquet(KEY_FILE)
df = annotated.merge(key, on="id", how="inner")

df["gold"] = pd.to_numeric(df["gold"], errors="coerce")
unannotated = int(df["gold"].isna().sum())
df = df.dropna(subset=["gold"]).copy()
df["gold"] = df["gold"].astype(int)
df["pred"] = (df["source"] == "pos").astype(int)

if unannotated > 0:
    print(f"WARNING: {unannotated} rows have no gold label and were skipped.")

P = precision_score(df["gold"], df["pred"], zero_division=0)
R = recall_score(df["gold"], df["pred"], zero_division=0)
F = f1_score(df["gold"], df["pred"], zero_division=0)
tn, fp, fn, tp = confusion_matrix(df["gold"], df["pred"], labels=[0, 1]).ravel()

print(f"N annotated:  {len(df)}")
print(f"Precision:    {P:.3f}   ({tp} TP / {tp + fp} predicted positives)")
print(f"Recall:       {R:.3f}   ({tp} TP / {tp + fn} actual positives)")
print(f"F1:           {F:.3f}")
print(f"Confusion:    TP={tp}  FP={fp}  TN={tn}  FN={fn}")

print("\n-- Up to 5 false positives (dict said AI, you said not) --")
for _, r in df[(df["pred"] == 1) & (df["gold"] == 0)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")
print("\n-- Up to 5 false negatives (you said AI, dict missed it) --")
for _, r in df[(df["pred"] == 0) & (df["gold"] == 1)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")

N annotated:  200
Precision:    1.000   (100 TP / 100 predicted positives)
Recall:       1.000   (100 TP / 100 actual positives)
F1:           1.000
Confusion:    TP=100  FP=0  TN=100  FN=0

-- Up to 5 false positives (dict said AI, you said not) --

-- Up to 5 false negatives (you said AI, dict missed it) --


## 4. Score with FinBERT-tone

Per-sentence cache is keyed on `sha256(sentence)`, so re-runs that
include overlapping sentences are nearly free.

In [ ]:
scored = None if FORCE_REFRESH else load_cached_step(INDICATOR, "scored")
if scored is None:
    if len(sentences_df) == 0:
        scored = sentences_df.assign(pos=0.0, neu=0.0, neg=0.0, label="neutral", confidence=0.0)
    else:
        scores = score_sentences(sentences_df["sentence"].tolist())
        scored = pd.concat(
            [sentences_df.reset_index(drop=True), scores[["pos", "neu", "neg", "label", "confidence"]]],
            axis=1,
        )
    save_cached_step(scored, INDICATOR, "scored")
    print(f"Scored {len(scored)} sentences (saved to {cache_path(INDICATOR, 'scored')})")
else:
    print(f"Loaded {len(scored)} scored sentences from cache ({cache_path(INDICATOR, 'scored')})")
scored.head()

## 5. Aggregate to firm level and write the indicator parquet

Produces two firm-level components following Babina, Fedyk, He,
Hodson (2024, JFE):

- `ai_sentence_share` — extensive margin (`n_ai_sentences / n_total_sentences`),
  length-normalized.
- `net_tone_finbert` — intensive margin (FinBERT-tone mean), set to
  `NaN` when `n_ai_sentences < MIN_AI_SENTENCES_FOR_TONE` (default 5,
  Huang/Wang/Yang 2023) because the 3-class mean is too noisy below
  that threshold for cross-firm comparison.

In [ ]:
indicator = aggregate_firm_level(filings, scored, sentence_totals_df)
print(f"Indicator rows: {len(indicator)}")
print(f"Firms with AI mentions:                       {int(indicator['has_ai_mention'].sum())}")
print(f"Firms above tone threshold (>= {MIN_AI_SENTENCES_FOR_TONE} AI sents): {int(indicator['net_tone_finbert'].notna().sum())}")
indicator.head()

## 6. Sanity checks

Per the plan's verification section: expect 480+ firms with a row,
~80% with `has_ai_mention=1`, `net_tone_finbert` (conditional on
threshold) with mean in [0.05, 0.20] and sd in [0.15, 0.25].

The "tone-threshold diagnostic" block reports the pass-rate of
`n_ai_sentences` at 3, 5, 10, 20. Use it to revisit
`MIN_AI_SENTENCES_FOR_TONE` empirically after the first full run:
if the pass-rate at 5 is below ~60% of AI-mentioning firms, lower
the threshold (e.g. to 3) in `aggregate.py`.

In [ ]:
share_with_ai = indicator["has_ai_mention"].mean()
n_with_ai = int(indicator["has_ai_mention"].sum())

print("=== Coverage ===")
print(f"  total firms:                        {len(indicator)}")
print(f"  share with AI mention:              {share_with_ai:.2%}  ({n_with_ai} firms)")

if n_with_ai > 0:
    tone_sub = indicator.loc[indicator["net_tone_finbert"].notna(), "net_tone_finbert"]
    print(f"  firms above tone threshold ({MIN_AI_SENTENCES_FOR_TONE}+):   {len(tone_sub)}  ({len(tone_sub)/len(indicator):.2%} of all, {len(tone_sub)/n_with_ai:.2%} of AI-mentioning)")
    print(f"  net_tone_finbert (above thr) mean:  {tone_sub.mean():+.3f}  sd: {tone_sub.std():.3f}")
    print(f"  ai_sentence_share (all)   mean:     {indicator['ai_sentence_share'].mean():.4f}  sd: {indicator['ai_sentence_share'].std():.4f}")

    print("\n=== Tone-threshold diagnostic ===")
    print("  (use to empirically revisit MIN_AI_SENTENCES_FOR_TONE)")
    n_ai = indicator["n_ai_sentences"]
    for thr in (3, 5, 10, 20):
        n_pass = int((n_ai >= thr).sum())
        print(f"  n_ai_sentences >= {thr:>2}: {n_pass:>3} firms  "
              f"({n_pass/len(indicator):.1%} of all, {n_pass/n_with_ai:.1%} of AI-mentioning)")

    print("\n=== n_ai_sentences distribution (AI-mentioning firms) ===")
    print(n_ai[n_ai > 0].describe().to_string())
else:
    print("  no firms with AI mentions — nothing to summarize")

indicator[
    [
        "ai_sentence_share",
        "net_tone_finbert",
        "net_tone_item_1",
        "net_tone_item_1a",
        "net_tone_item_7",
    ]
].describe()